# 01 — Preparação dos dados


Este notebook documenta o pipeline de dados. Ele **não** gera os arquivos: rode
antes, a partir da raiz do repositório, na ordem

```bash
python -m src.data.loader
python -m src.data.synthetic_generator
python -m src.data.curator
```

O papel aqui é evidenciar o que o pipeline produziu: estatísticas do PubMedQA, a
anonimização atuando, exemplos dos dados hospitalares sintéticos e o perfil do dataset
final de fine-tuning.

In [1]:
import sys
import warnings
from pathlib import Path

# O tqdm avisa que o ipywidgets nao esta instalado. E cosmetico (barra de progresso em
# texto em vez de widget) e o ipywidgets nao esta no requirements de proposito: o mirror
# interno usado pelo pip nao o serve, e adiciona-lo quebraria a instalacao do projeto.
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

# O notebook roda de dentro de notebooks/, então a raiz do repositório precisa entrar no
# sys.path — senão `import src...` falha com ModuleNotFoundError.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from collections import Counter

from src.data.anonymizer import anonymize
from src.data.loader import load_jsonl

print("raiz:", RAIZ)

raiz: /Users/ostrifezze/personal/POS/tech-challenge-group-24


## Célula 1 — PubMedQA: total e distribuição de labels

O subset `pqa_labeled` traz 1.000 perguntas clínicas com resposta longa revisada e um
label de decisão (`yes` / `no` / `maybe`).

In [2]:
from datasets import load_dataset

bruto = load_dataset("qiaojin/PubMedQA", "pqa_labeled")["train"]

print(f"Total de registros: {len(bruto)}")
print(f"Campos: {list(bruto.features)}")

labels = Counter(bruto["final_decision"])
print("\nDistribuição de labels:")
for label, quantidade in labels.most_common():
    barra = "█" * round(40 * quantidade / len(bruto))
    print(f"  {label:6} {quantidade:5}  {quantidade / len(bruto):5.1%}  {barra}")

palavras = [len(r.split()) for r in bruto["long_answer"]]
print(
    f"\nPalavras na resposta: min={min(palavras)} "
    f"media={sum(palavras) / len(palavras):.0f} max={max(palavras)}"
)
print(f"Respostas com menos de 20 palavras: {sum(1 for p in palavras if p < 20)}")

Total de registros: 1000
Campos: ['pubid', 'question', 'context', 'long_answer', 'final_decision']

Distribuição de labels:
  yes      552  55.2%  ██████████████████████
  no       338  33.8%  ██████████████
  maybe    110  11.0%  ████

Palavras na resposta: min=8 media=40 max=126
Respostas com menos de 20 palavras: 96


## Célula 2 — Anonimização: antes e depois

As regras são **ancoradas em contexto** (nome só sai quando precedido de `Dr.`, `paciente`,
`Sra.`). Uma regra genérica de "palavras capitalizadas = nome" destruiria termos
científicos do PubMedQA — o bloco final mede exatamente esse risco.

In [3]:
exemplos = [
    "Paciente atendido pelo Dr. João Carlos Silva em 12/03/2024.",
    "A paciente Maria Souza tem CPF 123.456.789-00 e telefone (11) 98765-4321.",
    "Contato: maria.souza@hospital.com.br, prontuário nº 4455667.",
    "Sr. Pedro Alves, CNS 123456789012345, retorno em 2024-05-01.",
]

for texto in exemplos:
    print(f"antes : {texto}")
    print(f"depois: {anonymize(texto)}\n")

print("=" * 70)
print("Texto científico que NÃO deve ser tocado:\n")
for texto in [
    "The study covered 2000-2012 in three centers.",
    "Mortality was lower (P = .04) in the treated group.",
    "Programmed Cell Death in Aponogeton madagascariensis.",
]:
    print(f"  {'intacto' if anonymize(texto) == texto else 'ALTERADO'} | {texto}")

pubmedqa = load_jsonl(RAIZ / "data" / "processed" / "pubmedqa.jsonl")
alterados = sum(
    1
    for r in pubmedqa
    for campo in ("instruction", "input", "output")
    if anonymize(r[campo]) != r[campo]
)
print(
    f"\nCampos alterados pela anonimização no PubMedQA: {alterados} de {len(pubmedqa) * 3}"
)
print("(o único caso contém datas reais — 8/1/97 — então a substituição é correta)")

antes : Paciente atendido pelo Dr. João Carlos Silva em 12/03/2024.
depois: Paciente atendido pelo [MÉDICO] em [DATA].

antes : A paciente Maria Souza tem CPF 123.456.789-00 e telefone (11) 98765-4321.
depois: A paciente [PACIENTE] tem CPF [PACIENTE_ID] e telefone [TELEFONE].

antes : Contato: maria.souza@hospital.com.br, prontuário nº 4455667.
depois: Contato: [EMAIL], prontuário nº [PACIENTE_ID].

antes : Sr. Pedro Alves, CNS 123456789012345, retorno em 2024-05-01.
depois: Sr. [PACIENTE], CNS [PACIENTE_ID], retorno em [DATA].

Texto científico que NÃO deve ser tocado:

  intacto | The study covered 2000-2012 in three centers.
  intacto | Mortality was lower (P = .04) in the treated group.
  intacto | Programmed Cell Death in Aponogeton madagascariensis.

Campos alterados pela anonimização no PubMedQA: 1 de 3000
(o único caso contém datas reais — 8/1/97 — então a substituição é correta)


## Célula 3 — Dados hospitalares sintéticos

Cobre os quatro tipos que o enunciado cita nominalmente: protocolos, perguntas frequentes
de médicos, e modelos de laudo, receita e procedimento interno. Todo o conteúdo é
fabricado por template — nenhum dado real.

In [4]:
sinteticos = load_jsonl(RAIZ / "data" / "synthetic" / "synthetic_hospital.jsonl")

tipos = Counter(r["source"].split(":")[0] for r in sinteticos)
print(f"Total: {len(sinteticos)} registros")
print(f"Tipos: {dict(tipos)}")
print(f"Condições CID-10 cobertas: {sorted({r['source'].split(':')[1] for r in sinteticos})}\n")

vistos = set()
for registro in sinteticos:
    tipo = registro["source"].split(":")[0]
    if tipo in vistos:
        continue
    vistos.add(tipo)
    print("=" * 70)
    print(f"[{registro['source']}]")
    print(f"instruction: {registro['instruction']}")
    print(f"input      : {registro['input']}")
    print(f"output     : {registro['output']}\n")

com_fonte = sum(1 for r in sinteticos if "[Fonte:" in r["output"])
com_disclaimer = sum(1 for r in sinteticos if "[Requer validação médica" in r["output"])
print("=" * 70)
print(f"Respostas que citam a fonte:        {com_fonte}/{len(sinteticos)}")
print(f"Respostas com validação humana:     {com_disclaimer}/{len(sinteticos)}")

Total: 100 registros
Tipos: {'Protocolo': 10, 'Laudo': 10, 'Receita': 10, 'Procedimento': 10, 'FAQ': 60}
Condições CID-10 cobertas: ['A09', 'E11', 'G43', 'I10', 'I21', 'J18', 'J44', 'J45', 'K29.7', 'N39.0']

[Protocolo:J45]
instruction: Descreva o protocolo institucional para asma.
input      : Condição: asma (CID-10 J45).
output     : Protocolo de asma (CID-10 J45). Avaliação: solicitar espirometria para confirmação diagnóstica. Conduta: broncodilatador inalatório de resgate e corticoide inalatório de manutenção. Procedimento associado: nebulização assistida. Registrar a evolução no prontuário do [PACIENTE] a cada reavaliação.

[Fonte: Protocolo:J45] [Requer validação médica por profissional habilitado]

[Laudo:J45]
instruction: Redija um modelo de laudo de espirometria.
input      : Exame: espirometria. Hipótese: asma (CID-10 J45).
output     : Laudo de espirometria realizado em [DATA]. Paciente: [PACIENTE]. Solicitante: [MÉDICO]. Achado: distúrbio ventilatório obstrutivo com respost

## Célula 4 — Dataset final de fine-tuning

`data/processed/dataset.jsonl` é o dataset consolidado que alimenta o fine-tuning.

In [5]:
dataset = load_jsonl(RAIZ / "data" / "processed" / "dataset.jsonl")

fontes = Counter(r["source"].split(":")[0] for r in dataset)
palavras = [len(r["output"].split()) for r in dataset]

print(f"Total de registros: {len(dataset)}")
print(f"  PubMedQA:   {len(pubmedqa)} lidos -> {fontes['PubMedQA']} no dataset final")
print(f"  Sintéticos: {len(sinteticos)} lidos -> {len(dataset) - fontes['PubMedQA']} no dataset final")

print("\nDistribuição por fonte:")
for fonte, quantidade in fontes.most_common():
    barra = "█" * max(1, round(40 * quantidade / len(dataset)))
    print(f"  {fonte:14} {quantidade:5}  {quantidade / len(dataset):5.1%}  {barra}")

print(
    f"\nTamanho da resposta: min={min(palavras)} "
    f"media={sum(palavras) / len(palavras):.0f} max={max(palavras)} palavras"
)
print(f"Perguntas únicas: {len({r['instruction'] for r in dataset})} de {len(dataset)}")

# O fine-tuning divide o dataset 90/10 de forma sequencial. O curator embaralha para
# que os registros hospitalares não caiam todos na validação.
corte = int(len(dataset) * 0.9)
hospital_treino = sum(1 for r in dataset[:corte] if not r["source"].startswith("PubMedQA"))
hospital_valid = sum(1 for r in dataset[corte:] if not r["source"].startswith("PubMedQA"))
print(
    f"\nSplit 90/10 do fine-tuning — registros hospitalares: "
    f"{hospital_treino} no treino, {hospital_valid} na validação"
)

Total de registros: 1004
  PubMedQA:   1000 lidos -> 904 no dataset final
  Sintéticos: 100 lidos -> 100 no dataset final

Distribuição por fonte:
  PubMedQA         904  90.0%  ████████████████████████████████████
  FAQ               60   6.0%  ██
  Procedimento      10   1.0%  █
  Laudo             10   1.0%  █
  Protocolo         10   1.0%  █
  Receita           10   1.0%  █

Tamanho da resposta: min=20 media=43 max=126 palavras
Perguntas únicas: 1004 de 1004

Split 90/10 do fine-tuning — registros hospitalares: 91 no treino, 9 na validação


---

Próxima etapa: **fine-tuning com MLX-LM**, que consome o `dataset.jsonl` acima.